# Build & Flash — MSPM0L2228 victim firmware

Compiles `mspm0l2228_aes_trigger.c` with the **MSPM0 SDK + arm-none-eabi-gcc**
(SysConfig runs automatically from the makefile) and flashes the result to the
board over your debug probe.

**Prerequisites (installed on this machine):**
- MSPM0 SDK, `arm-none-eabi-gcc`, and TI SysConfig (`sysconfig_cli`).
- A debug probe (XDS110 on the LaunchPad, or J-Link) and **one** flash tool:
  TI UniFlash (`dslite`), OpenOCD (with the `mspm0` target), or J-Link.
- **A project scaffold**: an SDK gcc makefile + linker + startup + a `.syscfg`
  that enables **AESADV**, one **UART** (115200), and one **GPIO output**
  (the trigger). Easiest is to copy an SDK driverlib example for
  `LP_MSPM0L2228`, open its `.syscfg` once in SysConfig to add those, then drop
  our `main.c` in. Section 3 has a helper to copy an example scaffold.

This notebook runs commands on *this* computer; the probe must be plugged in.
Every command is in an editable variable — adjust paths/filenames to your setup.


## 1. Parameters — set these to your machine

In [9]:
import os, glob, shutil, subprocess, shlex

# --- toolchain (paths confirmed on this machine) ---
SDK        = os.path.expanduser('~/Downloads/mspm0_sdk_2_11_00_07')          # MSPM0 SDK root
GCC        = '/usr'                                                          # /usr/bin/arm-none-eabi-gcc
SYSCONFIG  = '/home/gustavnybro/ti/ccs2101/ccs/utils/sysconfig_1.28.1/sysconfig_cli.sh'

# --- project (create this once in CCS: MSPM0L2228 DriverLib project with the
#     .syscfg set up for AESADV + UART_0 (115200) + GPIO_TRIG, then point here) ---
PROJECT_DIR = os.path.expanduser('~/mspm0-aes-victim')       # <-- your CCS project root
GCC_DIR     = os.path.join(PROJECT_DIR, 'gcc')               # where the gcc makefile lives
MAIN_SRC    = os.path.expanduser('~/Downloads/mspm0l2228_aes_trigger.c')
DEVICE      = 'MSPM0L2228'

# --- flashing ---
FLASH_METHOD = 'uniflash'

UNIFLASH_DIR = os.path.expanduser('~/ti/uniflash_9.6.0')
CCXML        = os.path.join(PROJECT_DIR, 'MSPM0L2228.ccxml')  # export once from UniFlash GUI

OOCD_IFACE  = 'interface/xds110.cfg'
OOCD_TARGET = 'target/mspm0.cfg'

JLINK_EXE = 'JLinkExe'

def run(cmd, cwd=None, env=None):
    print('$', cmd if isinstance(cmd, str) else ' '.join(cmd))
    r = subprocess.run(shlex.split(cmd) if isinstance(cmd, str) else cmd,
                       cwd=cwd, env=env, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0:
        if r.stderr: print('--- STDERR ---\n', r.stderr[-4000:])
    return r

## 2. Check the toolchain is present

Fails early with a clear message rather than a confusing build error later.

In [10]:
gcc_bin = os.path.join(GCC, 'bin', 'arm-none-eabi-gcc')
checks = {
    'SDK dir': os.path.isdir(SDK),
    'arm-none-eabi-gcc': os.path.isfile(gcc_bin),
    'sysconfig_cli': os.path.isfile(SYSCONFIG),
    'project dir': os.path.isdir(PROJECT_DIR),
    'firmware main.c': os.path.isfile(MAIN_SRC),
}
for k, v in checks.items():
    print(('OK  ' if v else 'MISSING  ') + k)

which = lambda t: shutil.which(t) or '(not on PATH)'
print('\nflash tools on PATH:')
print('  openocd :', which('openocd'))
print('  dslite  :', which('dslite') or os.path.join(UNIFLASH_DIR, 'dslite.sh'))
print('  JLinkExe:', which('JLinkExe'))

OK  SDK dir
OK  arm-none-eabi-gcc
OK  sysconfig_cli
MISSING  project dir
OK  firmware main.c

flash tools on PATH:
  openocd : (not on PATH)
  dslite  : (not on PATH)
  JLinkExe: (not on PATH)


## 3. (Optional) Copy an SDK example scaffold, then our main.c

Skip if `PROJECT_DIR` already has a working gcc makefile + `.syscfg`. Otherwise
this copies an SDK driverlib example for the LaunchPad as a starting scaffold —
then open its `.syscfg` **once** in SysConfig to enable AESADV, a UART at 115200,
and a GPIO output trigger, before building.

In [ ]:
EXAMPLE = os.path.join(SDK, 'examples/nortos/LP_MSPM0L2228/driverlib/gpio_toggle_output')

if not os.path.isdir(PROJECT_DIR):
    if os.path.isdir(EXAMPLE):
        shutil.copytree(EXAMPLE, PROJECT_DIR)
        print('Copied scaffold from', EXAMPLE)
    else:
        print('Example not found — check SDK path / pick another example under',
              os.path.join(SDK, 'examples/nortos/LP_MSPM0L2228/driverlib'))
else:
    print('PROJECT_DIR already exists — not overwriting.')

# Drop our firmware in as the project's main source (adjust the target name if the
# scaffold's main file is named differently).
if os.path.isdir(PROJECT_DIR) and os.path.isfile(MAIN_SRC):
    dst = None
    for cand in ('gpio_toggle_output.c', 'main.c', 'empty.c'):
        p = os.path.join(PROJECT_DIR, cand)
        if os.path.isfile(p): dst = p; break
    dst = dst or os.path.join(PROJECT_DIR, 'main.c')
    shutil.copyfile(MAIN_SRC, dst)
    print('Placed firmware at', dst)
    print('\n>> Now open the .syscfg in SysConfig and add AESADV + UART_0 (115200) + GPIO_TRIG,')
    print('   matching the labels in the firmware header, then run section 4.')

## 4. Build

Runs the SDK gcc makefile; it invokes SysConfig to regenerate
`ti_msp_dl_config.*` and compiles to an ELF (`.out`). Tool paths are passed on
the command line so you don't have to edit `imports.mak`.

In [ ]:
env = dict(os.environ)
env['PATH'] = os.path.join(GCC, 'bin') + os.pathsep + env['PATH']

r = run(f'make -j GCC_ARMCOMPILER={shlex.quote(GCC)} '
        f'SYSCONFIG_TOOL={shlex.quote(SYSCONFIG)}',
        cwd=GCC_DIR, env=env)

outs = sorted(glob.glob(os.path.join(GCC_DIR, '**', '*.out'), recursive=True),
              key=os.path.getmtime)
assert r.returncode == 0 and outs, 'Build failed — see output above (check SDK/gcc/sysconfig paths, and that the .syscfg has AESADV/UART/GPIO).'
IMAGE = outs[-1]
print('\nBuilt:', IMAGE)

## 5. Flash

Program, verify, reset, and run — via the tool you selected in section 1.

In [ ]:
def flash_uniflash():
    dslite = shutil.which('dslite') or os.path.join(UNIFLASH_DIR, 'dslite.sh')
    return run(f'{shlex.quote(dslite)} --config {shlex.quote(CCXML)} --verbose {shlex.quote(IMAGE)}')

def flash_openocd():
    return run(f'openocd -f {OOCD_IFACE} -f {OOCD_TARGET} '
               f'-c {shlex.quote(f"program {IMAGE} verify reset exit")}')

def flash_jlink():
    script = os.path.join(PROJECT_DIR, '_flash.jlink')
    with open(script, 'w') as f:
        f.write(f'device {DEVICE}\nsi SWD\nspeed 4000\n'
                f'loadfile {IMAGE}\nr\ng\nqc\n')
    return run(f'{shlex.quote(JLINK_EXE)} -CommandFile {shlex.quote(script)}')

r = {'uniflash': flash_uniflash, 'openocd': flash_openocd, 'jlink': flash_jlink}[FLASH_METHOD]()
print('\nFlash', 'OK' if r.returncode == 0 else 'FAILED (see above)')

## 6. After flashing

The MSPM0 resets and runs the victim: it waits for `'p'`+16 plaintext bytes and
replies `'r'`+16 ciphertext bytes, pulsing the trigger around the AES-ADV block.
Verify the link and trigger from **section 4 of `mspm0_hwaes_cpa.ipynb`** (that's
where the serial goes through the Husky), then capture and attack.

## Troubleshooting
- **`make` can't find SysConfig / product.json:** check `SYSCONFIG` points at
  `sysconfig_cli.sh` and `SDK` is the SDK root; some makefiles also want
  `SYSCONFIG_TOOL` as a dir — if so, pass the folder instead of the script.
- **Compile errors on `DL_AESADV_*` or `GPIO_TRIG_*`/`UART_0_INST`:** the
  `.syscfg` doesn't define those peripherals/labels yet, or your SDK version
  names a signature differently — open SysConfig and match the labels in the
  firmware header.
- **OpenOCD `could not find target/interface cfg`:** the file names differ in
  your OpenOCD build. List your scripts dir and set `OOCD_IFACE` / `OOCD_TARGET`
  to the actual XDS110 and MSPM0 cfgs (or use `-c "adapter driver xds110"`).
- **dslite: no ccxml / wrong device:** create the connection once in the UniFlash
  GUI (XDS110 + MSPM0L2228), export the `.ccxml`, point `CCXML` at it.
- **Probe not found:** check the USB cable/permissions (Linux udev rules for
  XDS110/J-Link) and that no other tool (CCS, a debug session) holds the probe.
